In [9]:
import pandas as pd
import datetime
from datetime import timezone

# Overview

activity_ts1.csv has a little over 2 million rows. The data is workout heart rate data for twenty five individuals. Each row has a heart rate measurement which is the average heart rate over the interval from the current time to the next time in the time series. Columns:

- ‘userName’ = unique identifier for each user
- ‘timeUTCsecs’ = time of the measurement, in the UTC time zone, in seconds
- ‘heartRate’ = average heartrate from the interval timeUTCsecs to the next time
- ‘timeLocalSecs’ = time of the measurement, in the local time zone, in seconds
- ‘summaryId’ = unique identifier for each workout
- ‘activityType’ = workout type

meta.csv is metadata on each user in the first csv. It contains a mapping of username to age and gender. Columns:

- ‘userName’ = unique identifier for each user
- ‘DOB = user’s birthday
- ‘gender’ = user’s gender 

# Task

Calculate time in heart rate zones for each workout. This should result in a table whose index is ‘summaryId’ and columns are [‘zone1’, ’zone2’, ’zone3’, ’zone4’, ’zone5’, ’zone_6’]. The values of the table are total time in each zone for the given workout row. 

# Order of operations 

1. Read in the data
2. Check for NaNs and decide how to fix them
3. Calculate the age of each user, and their corresponding estimated maximum heart rate. 
4. Join the maximum heart rate information to the workout data table. 
5. Compute what zone each row of data in the workout table belongs to. 
6. Compute the duration for each row of data in the workout table, assuming that the next row indicates the next time if the next row belongs in the same workout, and assume a duration of 1 second otherwise. 
7. Clean the table, dropping rows that do not correspond to one of the 6 zones, and dropping the columns that we don't need. 
8. Group by the workout and the zone, sum, and then fix the column and row indices.

### Reading in the data 

In [6]:
workout_data = pd.read_excel('data.xlsx')
workout_data.head()

,userName,timeUTCsecs,heartRate,timeLOCALSecs,summaryId,activityType
0,user_21,1668866696,73.0,1668848696,10002964730-detail,LAP_SWIMMING
1,user_21,1668866697,72.0,1668848697,10002964730-detail,LAP_SWIMMING
2,user_21,1668866698,72.0,1668848698,10002964730-detail,LAP_SWIMMING
3,user_21,1668866699,72.0,1668848699,10002964730-detail,LAP_SWIMMING
4,user_21,1668866700,73.0,1668848700,10002964730-detail,LAP_SWIMMING


In [7]:
user_data = pd.read_csv('meta.csv')
user_data.head()

,userName,DOB,gender
0,user_0,2003-09-05,female
1,user_1,2003-01-20,female
2,user_2,1961-06-28,female
3,user_3,1965-11-11,female
4,user_4,1993-08-10,male


# Check the data is nicely behaved

In [ ]:
user_data.isna().any().any() # no NA values in user_data

False

In [ ]:
workout_data.isna().any().any() # ope

True

In [54]:
# workout_data.isna() gives a df of the same shape with True/False of whether the value is NaN, and then .any(axis=1) applies column wise to see if any columns in a given row are NaN
rows_with_nan = workout_data[workout_data.isna().any(axis=1)] 
rows_with_nan

,userName,timeUTCsecs,heartRate,timeLOCALSecs,summaryId,activityType
610017,user_14,1690481119,NaN,1690445119,11656391053-detail,HIKING
610037,user_14,1690481176,NaN,1690445176,11656391053-detail,HIKING
702670,user_4,1693094957,NaN,1693069757,11881550948-detail,OTHER
702948,user_4,1693096201,NaN,1693071001,11881550948-detail,OTHER


In [145]:
for index in rows_with_nan.index:
    # before and after all exist and are numbers that make sense
    print(f"Before NaN: {workout_data.iloc[index - 1]['heartRate']}, after NaN: {workout_data.iloc[index + 1]['heartRate']}")
    # befoer and after belong from the same workout, so it makes sense to interpolate
    print(f"Before workout: {workout_data.iloc[index  - 1]['summaryId']}, after workout: {workout_data.iloc[index + 1]['summaryId']}")
    assert workout_data.iloc[index - 1]['summaryId'] == workout_data.iloc[index + 1]['summaryId']

Before NaN: 95.0, after NaN: 97.0
Before workout: 11656391053-detail, after workout: 11656391053-detail
Before NaN: 94.0, after NaN: 97.0
Before workout: 11656391053-detail, after workout: 11656391053-detail
Before NaN: 101.0, after NaN: 104.0
Before workout: 11881550948-detail, after workout: 11881550948-detail
Before NaN: 77.0, after NaN: 85.0
Before workout: 11881550948-detail, after workout: 11881550948-detail


In [71]:
# these all look fairly regular, so here we choose to interpolate between the heart rates before and after the NaN
for index in rows_with_nan.index:
    before = workout_data.iloc[index - 1]['heartRate']
    after = workout_data.iloc[index + 1]['heartRate']
    interpolation = (before + after) / 2
    workout_data.loc[index, 'heartRate'] = interpolation

In [ ]:
# now check that there are no more NaNs in the workout_data table
workout_data.isna().any().any() # nice!

False

# Estimating maximum heart rate using the metadata

### Aside: Time zones

In essence, the date of birth given does not have a time zone attached to it, like the workout time does. This means that the calculation of a user's age might be off by a little bit. As a user of health/workout apps myself, no one asks for information about the time zone I was born in, so I assume this does not matter in practice, and I will be using UTC time to calculate the ages of the users.

There is an argument to be made to calculating the user's age relative to their local time zone, which can be worked out via the `timeUTCSecs` and `timeLOCALSecs` columns from the workout data table, but who says people don't move from the place they were born?  

In [22]:
# documentation for this computation used: https://docs.python.org/3/library/datetime.html#datetime-objects
def compute_age(dob: str) -> int: 
    """Parses the date of birth of the individual, and returns their age in years."""
    y, m, d = dob.split("-")
    datetime_dob = datetime.datetime(int(y), int(m), int(d), tzinfo=timezone.utc)
    diff = datetime.datetime.now(timezone.utc) - datetime_dob
    return diff.days // 365

In [24]:
user_data['age'] = user_data['DOB'].apply(compute_age)
user_data.head()

,userName,DOB,gender,age
0,user_0,2003-09-05,female,22
1,user_1,2003-01-20,female,23
2,user_2,1961-06-28,female,64
3,user_3,1965-11-11,female,60
4,user_4,1993-08-10,male,32


In [ ]:
def compute_max_hr(row: pd.Series) -> float:
    """
    Computes the maximum estimated heart rate of a row of the user_data table.
    Males: 208.609 - 0.716 x age
    Females: 209.273 - 0.804 x age 
    """
    if row['gender'] == 'female':
        return 209.273 - 0.804 * row['age']
    elif row['gender'] == 'male':
        return 208.609 - 0.716 * row['age']
    else: 
        raise Exception()

In [33]:
user_data['max_hr'] = user_data.apply(compute_max_hr, axis=1)
user_data.head()

,userName,DOB,gender,age,max_hr
0,user_0,2003-09-05,female,22,191.585
1,user_1,2003-01-20,female,23,190.781
2,user_2,1961-06-28,female,64,157.817
3,user_3,1965-11-11,female,60,161.033
4,user_4,1993-08-10,male,32,185.697


# Determine zone for each of the chunks of excercise

First, join over the maximum heart rate information from the `user_data` table to the `workout_data` table

In [165]:
data = pd.merge(workout_data, user_data, 'left', on='userName')
data.head()

,userName,timeUTCsecs,heartRate,timeLOCALSecs,summaryId,activityType,DOB,gender,age,max_hr
0,user_21,1668866696,73.0,1668848696,10002964730-detail,LAP_SWIMMING,1971-04-11,male,54,169.945
1,user_21,1668866697,72.0,1668848697,10002964730-detail,LAP_SWIMMING,1971-04-11,male,54,169.945
2,user_21,1668866698,72.0,1668848698,10002964730-detail,LAP_SWIMMING,1971-04-11,male,54,169.945
3,user_21,1668866699,72.0,1668848699,10002964730-detail,LAP_SWIMMING,1971-04-11,male,54,169.945
4,user_21,1668866700,73.0,1668848700,10002964730-detail,LAP_SWIMMING,1971-04-11,male,54,169.945


In [166]:
# this assumes that there is no heart rate zone before zone 1 that we care about, so we output pd.NA and drop the rows
def determine_zone(row: pd.Series) -> str:
    """Determines the heart rate zone for a row of the joined data table.
    Zone 1: 50-60% of max
    Zone 2: 60-70% of max
    Zone 3: 70-80% of max
    Zone 4: 80-90% of max
    Zone 5: 90-100% of max
    Zone 6: >100% of max
    """
    percentage = row['heartRate'] / row['max_hr'] * 100
    if percentage > 100:
        return "zone 6"
    elif percentage > 90:
        return "zone 5"
    elif percentage > 80:
        return "zone 4"
    elif percentage > 70:
        return "zone 3"
    elif percentage > 60:
        return "zone 2"
    elif percentage > 50:
        return "zone 1"
    else:
        return pd.NA

In [167]:
data['zone'] = data.apply(determine_zone, axis=1)

### Aside: Checking that all workout times are continuous in increments of 1 second
We would like to count time by saying that each row in a heart rate zone corresponds to exactly one second of excercise, as from inspecting the measurements this seems to be true. Devoid of this assumption we would have to actually take the difference between the times indicated in every 2 rows, and also have to think harder about what to do for the end of a workout, where there doesn't seem a row that only indicates the end time.

In [168]:
def in_increments_of_one(l: list[int]):
    """Returns True if the list given increments in steps of 1"""
    return l == list(range(l[0], len(l) + l[0], 1))

In [169]:
# some quick sanity checks 
assert in_increments_of_one([1, 2, 3]) == True
assert in_increments_of_one([1, 2, 4]) == False
assert in_increments_of_one([1, 4, 3]) == False

In [170]:
for group, df in data.groupby("summaryId"):
    current_list = list(df['timeUTCsecs'])
    try:
        assert in_increments_of_one(current_list)
    except:
        non_increment_list = current_list
        break

In [171]:
in_increments_of_one(non_increment_list[700:715])

False

In [172]:
print(non_increment_list[700:715])

[1668867396, 1668867397, 1668867398, 1668867399, 1668867400, 1668867401, 1668867402, 1668867403, 1668867404, 1668867580, 1668867581, 1668867582, 1668867583, 1668867584, 1668867585]


The non-continuous part appears at 1668867403, 1668867404, 1668867580, 1668867581 above

This means we will have to calculate time by looking at start time from the current row and the end time at the next row

For the purpose of the analysis, we will just assume that the last row from a given workout ends one second after. 

# Calculating amount of time elapsed for each row

In [173]:
def determine_time_elapsed(df: pd.DataFrame, row: pd.Series):
    index = row.name # which row is it?
    
    if len(df) <= index + 1:
        # next row does not exist, we have hit the end of the dataframe, so we assume elapsed time is 1 second
        return 1
    
    curr_workout = row['summaryId']
    next_row = df.iloc[index + 1]
    next_workout = next_row['summaryId']
    
    if curr_workout != next_workout:
        # here we assume that the workout is over, and given no explicit endtime we will assume a final elapsed time of 1 second
        return 1 
    else:
        return next_row['timeUTCsecs'] - row['timeUTCsecs']    

In [174]:
data['time_elapsed'] = data.apply(lambda x: determine_time_elapsed(data, x), axis=1)

In [175]:
data.head()

,userName,timeUTCsecs,heartRate,timeLOCALSecs,summaryId,activityType,DOB,gender,age,max_hr,zone,time_elapsed
0,user_21,1668866696,73.0,1668848696,10002964730-detail,LAP_SWIMMING,1971-04-11,male,54,169.945,<NA>,1
1,user_21,1668866697,72.0,1668848697,10002964730-detail,LAP_SWIMMING,1971-04-11,male,54,169.945,<NA>,1
2,user_21,1668866698,72.0,1668848698,10002964730-detail,LAP_SWIMMING,1971-04-11,male,54,169.945,<NA>,1
3,user_21,1668866699,72.0,1668848699,10002964730-detail,LAP_SWIMMING,1971-04-11,male,54,169.945,<NA>,1
4,user_21,1668866700,73.0,1668848700,10002964730-detail,LAP_SWIMMING,1971-04-11,male,54,169.945,<NA>,1


# Doing some cleaning up

In [177]:
cleaned_data = data.drop(columns=['timeUTCsecs', 'timeLOCALSecs', 'activityType', 'DOB', 'gender', 'age', 'max_hr', 'heartRate', 'userName'])
cleaned_data.dropna(inplace=True) # drop the rows corresponding to zones we don't care about
cleaned_data.head()

,summaryId,zone,time_elapsed
22,10002964730-detail,zone 1,1
23,10002964730-detail,zone 1,1
24,10002964730-detail,zone 1,1
25,10002964730-detail,zone 1,1
26,10002964730-detail,zone 1,1


# Grouping by and summing up time elapsed

Using https://pandas.pydata.org/pandas-docs/stable/user_guide/groupby.html as a reference for this part.

In [233]:
grouped = cleaned_data.groupby(['summaryId', 'zone']).sum()

In [234]:
grouped = grouped.unstack('zone') # unstacking the zone and making it columns
grouped

time_elapsed                                        
zone                     zone 1  zone 2  zone 3  zone 4  zone 5  zone 6
summaryId                                                              
10002964730-detail       1172.0   736.0  1288.0   460.0     NaN     NaN
10008297393-detail        975.0   842.0     NaN     NaN     NaN     NaN
10013215888-detail        136.0   357.0   780.0   464.0    67.0     NaN
10016099102-detail         58.0  1167.0   864.0     NaN     NaN     NaN
10018607537-detail        329.0   475.0  1245.0  1282.0   451.0     1.0
...                         ...     ...     ...     ...     ...     ...
13180270394-detail         30.0    61.0   373.0  1644.0     NaN     NaN
13191598228-detail         86.0   384.0  2192.0    71.0     NaN     NaN
13191845624-detail          NaN   160.0    25.0     NaN     NaN     NaN
13191873870-detail        317.0   664.0   789.0  1423.0  2562.0  1419.0
13196363158-detail          NaN   157.0    49.0     6.0     NaN     NaN

[985 rows x 6 columns]

In [235]:
grouped.columns # multi-index looks weird and is not what i want

MultiIndex([('time_elapsed', 'zone 1'),
            ('time_elapsed', 'zone 2'),
            ('time_elapsed', 'zone 3'),
            ('time_elapsed', 'zone 4'),
            ('time_elapsed', 'zone 5'),
            ('time_elapsed', 'zone 6')],
           names=[None, 'zone'])

In [236]:
grouped.columns = grouped.columns.droplevel(0)
grouped

zone,zone 1,zone 2,zone 3,zone 4,zone 5,zone 6
summaryId,,,,,,
10002964730-detail,1172.0,736.0,1288.0,460.0,NaN,NaN
10008297393-detail,975.0,842.0,NaN,NaN,NaN,NaN
10013215888-detail,136.0,357.0,780.0,464.0,67.0,NaN
10016099102-detail,58.0,1167.0,864.0,NaN,NaN,NaN
10018607537-detail,329.0,475.0,1245.0,1282.0,451.0,1.0
...,...,...,...,...,...,...
13180270394-detail,30.0,61.0,373.0,1644.0,NaN,NaN
13191598228-detail,86.0,384.0,2192.0,71.0,NaN,NaN
13191845624-detail,NaN,160.0,25.0,NaN,NaN,NaN


In [237]:
# now just doing some cleanup so we don't have the name of the columns/rows indeces
grouped.columns.name = None
grouped.index.name = None

In [238]:
# now fill the na values with 0, these are the zones that didn't have any rows, just means that the person didn't spend time in that zone
grouped.fillna(0.0)

,zone 1,zone 2,zone 3,zone 4,zone 5,zone 6
10002964730-detail,1172.0,736.0,1288.0,460.0,0.0,0.0
10008297393-detail,975.0,842.0,0.0,0.0,0.0,0.0
10013215888-detail,136.0,357.0,780.0,464.0,67.0,0.0
10016099102-detail,58.0,1167.0,864.0,0.0,0.0,0.0
10018607537-detail,329.0,475.0,1245.0,1282.0,451.0,1.0
...,...,...,...,...,...,...
13180270394-detail,30.0,61.0,373.0,1644.0,0.0,0.0
13191598228-detail,86.0,384.0,2192.0,71.0,0.0,0.0
13191845624-detail,0.0,160.0,25.0,0.0,0.0,0.0
13191873870-detail,317.0,664.0,789.0,1423.0,2562.0,1419.0
